# AMERS v4 — Model Testing & Live Demo

**Standalone notebook** for testing the trained v4 model.

- Loads all trained checkpoints from Google Drive
- Picks random test samples and shows predictions with confidence
- Generates bar charts, confusion matrix, and per-class stats

**Prerequisites:** Run the full training pipeline in `00_setup_and_run.ipynb` first.

## 1. Setup — Mount Drive, Clone Repo, Fix Dependencies

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  DOWNLOAD CHECKPOINTS from Google Drive (bypass mount issues) ║
# ║  Only downloads files that are missing locally                ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

!pip install -q gdown

CKPT_BASE = '/content/drive/MyDrive/AMERS/outputs/checkpoints'

# Google Drive file IDs → local paths
FILES = {
    # v3 checkpoints
    '1sQtw7Om8s_veY6SzOHGPv4Q6qjVUGAHi': f'{CKPT_BASE}/v3/eeg_encoder_contrastive.pt',
    '1lY4L0gZDFlk2zHYezYEvZHecmCmoMBvI': f'{CKPT_BASE}/v3/speech_encoder_contrastive.pt',
    '1D4p3p_tKKss2znA5e8JHJ0tgw7I3h0pc': f'{CKPT_BASE}/v3/eeg_encoder_dann.pt',
    '1NDkyw21q2V_luvO-8X3zqEHqw-s52IY-': f'{CKPT_BASE}/v3/speech_encoder_dann.pt',
    '1hzU6fMF1FOPe1iVUiS8YY9LrwwL-0swG': f'{CKPT_BASE}/v3/best_transformer_fusion.pt',
    # Base checkpoints
    '1Dn44orMmVHQxQKPfWOBslWUEgktB8F1c': f'{CKPT_BASE}/gan/gan_final.pt',
    '1u0R_okolItQlIgx3Xp3U6uyrr9UW7RI8': f'{CKPT_BASE}/speech/speech_encoder_final.pt',
    '14xKckanol4h1JUIEM16geS8uJF8cDins': f'{CKPT_BASE}/eeg/eeg_encoder_final.pt',
}

downloaded, skipped = 0, 0
for file_id, dest in FILES.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f'  ✅ Already exists: {os.path.basename(dest)}')
        skipped += 1
    else:
        print(f'  ⬇️  Downloading: {os.path.basename(dest)}...')
        !gdown --id {file_id} -O {dest} -q
        if os.path.exists(dest):
            sz = os.path.getsize(dest) / (1024*1024)
            print(f'      ✅ Done ({sz:.1f} MB)')
            downloaded += 1
        else:
            print(f'      ❌ FAILED')

print(f'\n✅ {downloaded} downloaded, {skipped} already existed')

In [6]:
!git -C /content/amers pull --ff-only

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 7), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 11.45 KiB | 146.00 KiB/s, done.
From https://github.com/RAVINDRA8008/MAJORDRAFT
   61faea2..2036d03  main       -> origin/main
Updating 61faea2..2036d03
Fast-forward
 notebooks/00_setup_and_run.ipynb    | 336 ++----------------
 notebooks/06_results_analysis.ipynb |  32 +-
 notebooks/07_test_demo.ipynb        | 681 ++++++++++++++++++++++++++++++++++++
 3 files changed, 739 insertions(+), 310 deletions(-)
 create mode 100644 notebooks/07_test_demo.ipynb


In [16]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 DEBUG — Find where checkpoints actually are on Drive     ║
# ║  DELETE THIS CELL after you find the right paths!             ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

BASE = '/content/drive/MyDrive'
SEARCH_ROOTS = [
    f'{BASE}/AMERS',
    f'{BASE}',
]

print("🔍 Searching for .pt checkpoint files on Drive...\n")

found = []
for root_dir in SEARCH_ROOTS:
    if not os.path.exists(root_dir):
        print(f"  ❌ {root_dir} — does not exist")
        continue
    print(f"  📂 Scanning: {root_dir}")
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for f in filenames:
            if f.endswith('.pt') or f.endswith('.pth'):
                full = os.path.join(dirpath, f)
                sz = os.path.getsize(full) / 1024
                found.append((full, sz))

if found:
    print(f"\n✅ Found {len(found)} checkpoint files:\n")
    for path, sz in sorted(found):
        print(f"  {sz:8.0f} KB  {path}")
else:
    print("\n❌ No .pt/.pth files found!")

# Also show the directory structure of AMERS
print(f"\n{'='*60}")
print("📁 AMERS directory structure:")
print(f"{'='*60}")
amers_root = f'{BASE}/AMERS'
if os.path.exists(amers_root):
    for dirpath, dirnames, filenames in os.walk(amers_root):
        depth = dirpath.replace(amers_root, '').count(os.sep)
        indent = '  ' * depth
        print(f"{indent}📂 {os.path.basename(dirpath)}/")
        sub_indent = '  ' * (depth + 1)
        for f in filenames[:20]:  # limit files shown per dir
            print(f"{sub_indent}📄 {f}")
        if len(filenames) > 20:
            print(f"{sub_indent}... and {len(filenames)-20} more files")
else:
    print(f"  ❌ {amers_root} does not exist!")
    # Try finding any AMERS-like folder
    print(f"\n  Searching for 'AMERS' or 'amers' anywhere in Drive...")
    for dirpath, dirnames, _ in os.walk(BASE):
        for d in dirnames:
            if 'amers' in d.lower():
                print(f"  📂 Found: {os.path.join(dirpath, d)}")
        if dirpath.count(os.sep) > 5:  # don't go too deep
            break

🔍 Searching for .pt checkpoint files on Drive...

  📂 Scanning: /content/drive/MyDrive/AMERS
  📂 Scanning: /content/drive/MyDrive

❌ No .pt/.pth files found!

📁 AMERS directory structure:
📂 AMERS/
  📂 logs/
  📂 outputs/
    📂 checkpoints/
      📂 eeg/
      📂 v3/
      📂 rl/
      📂 speech/
      📂 gan/
      📂 fusion/
  📂 data/
    📂 deap/
      📂 processed/
      📂 raw/
    📂 iemocap/
      📂 processed/
      📂 raw/


In [13]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SETUP — Mount Drive, clone repo, fix numpy/scipy if needed   ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess
from pathlib import Path

# ── Robust Drive mount: verify it's truly accessible ──
from google.colab import drive

drive_ok = False
try:
    if os.path.exists('/content/drive/MyDrive'):
        os.listdir('/content/drive/MyDrive')
        drive_ok = True
        print('Drive already mounted and accessible ✅')
except OSError:
    pass

if not drive_ok:
    # Clean up stale/broken mount before re-mounting
    print('Drive not accessible — cleaning up and mounting...')
    subprocess.run(['fusermount', '-u', '/content/drive'],
                   capture_output=True, timeout=10)
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
    print('Drive mounted ✅')

# Clone or pull repo
REPO_DIR = '/content/amers'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RAVINDRA8008/MAJORDRAFT.git {REPO_DIR}
    print('Repository cloned.')
else:
    !git -C {REPO_DIR} pull --ff-only
    print('Repository updated.')

os.chdir(REPO_DIR)

# Fix numpy/scipy compatibility (Colab sometimes has mismatched versions)
print('\nFixing numpy/scipy compatibility...')
!pip install -q --upgrade numpy scipy scikit-learn 2>&1 | tail -3

# Install project deps
!pip install -q -r requirements.txt 2>&1 | tail -3
print('\n✅ Dependencies ready.')

# Verify
import numpy as np
import torch
print(f'NumPy {np.__version__}, PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU — will run on CPU (slower but works)')

Drive already mounted and accessible ✅
Already up to date.
Repository updated.

Fixing numpy/scipy compatibility...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.

✅ Dependencies ready.
NumPy 1.26.4, PyTorch 2.10.0+cu128
GPU: NVIDIA L4


In [19]:
import subprocess
subprocess.run(['ls', '-la', '/content/drive/MyDrive/AMERS/outputs/checkpoints/v3/'])

CompletedProcess(args=['ls', '-la', '/content/drive/MyDrive/AMERS/outputs/checkpoints/v3/'], returncode=0)

In [20]:
import os

BASE = '/content/drive/MyDrive'
print("🔍 Searching for .pt checkpoint files on Drive...\n")

found = []
for dirpath, dirnames, filenames in os.walk(BASE):
    for f in filenames:
        if f.endswith('.pt') or f.endswith('.pth'):
            full = os.path.join(dirpath, f)
            sz = os.path.getsize(full) / 1024
            found.append((full, sz))
    if dirpath.count(os.sep) > 8:
        break

if found:
    print(f"✅ Found {len(found)} checkpoint files:\n")
    for path, sz in sorted(found):
        print(f"  {sz:8.0f} KB  {path}")
else:
    print("❌ No .pt/.pth files found!")

print(f"\n{'='*60}")
print("📁 AMERS folder search:")
for dirpath, dirnames, _ in os.walk(BASE):
    for d in dirnames:
        if 'amers' in d.lower() or 'checkpoint' in d.lower():
            full = os.path.join(dirpath, d)
            print(f"  📂 {full}")
            for sub, _, files in os.walk(full):
                for f in files[:10]:
                    print(f"      📄 {os.path.join(sub, f)}")
    if dirpath.count(os.sep) > 5:
        break

🔍 Searching for .pt checkpoint files on Drive...

❌ No .pt/.pth files found!

📁 AMERS folder search:
  📂 /content/drive/MyDrive/AMERS
  📂 /content/drive/MyDrive/AMERS/outputs/checkpoints


## 2. Verify Checkpoints Exist

In [21]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CHECK: Are all required checkpoints on Drive?                ║
# ╚══════════════════════════════════════════════════════════════════╝
from pathlib import Path

CKPT = Path('/content/drive/MyDrive/AMERS/outputs/checkpoints')

required = {
    'EEG Encoder (v2)':      'eeg/eeg_encoder_final.pt',
    'Speech Encoder (v2)':   'speech/speech_encoder_final.pt',
    'GAN (WGAN-GP)':         'gan/gan_final.pt',
    'EEG Contrastive (v3)':  'v3/eeg_encoder_contrastive.pt',
    'Speech Contrastive (v3)':'v3/speech_encoder_contrastive.pt',
    'EEG DANN (v3)':         'v3/eeg_encoder_dann.pt',
    'Speech DANN (v3)':      'v3/speech_encoder_dann.pt',
    'Transformer Fusion':    'v3/best_transformer_fusion.pt',
}

print('Checkpoint Status:')
print('=' * 55)
all_ok = True
for label, path in required.items():
    p = CKPT / path
    if p.exists():
        sz = p.stat().st_size / 1024
        print(f'  ✅ {label:30s} ({sz:.0f} KB)')
    else:
        print(f'  ❌ {label:30s} — MISSING')
        all_ok = False
print('=' * 55)

if all_ok:
    print('\n✅ All checkpoints found — ready to test!')
else:
    print('\n⚠️ Some checkpoints missing. The model will use the best available.')
    print('  Run 00_setup_and_run.ipynb to train missing components.')

Checkpoint Status:
  ❌ EEG Encoder (v2)               — MISSING
  ❌ Speech Encoder (v2)            — MISSING
  ❌ GAN (WGAN-GP)                  — MISSING
  ❌ EEG Contrastive (v3)           — MISSING
  ❌ Speech Contrastive (v3)        — MISSING
  ❌ EEG DANN (v3)                  — MISSING
  ❌ Speech DANN (v3)               — MISSING
  ❌ Transformer Fusion             — MISSING

⚠️ Some checkpoints missing. The model will use the best available.
  Run 00_setup_and_run.ipynb to train missing components.


## 3. Load Trained Models

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  LOAD ALL TRAINED v4 MODELS                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys, torch
from pathlib import Path

sys.path.insert(0, '/content/amers')

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.utils.paths import get_paths
from src.utils.device import get_device
from src.models.eeg_encoder import EEGEncoder
from src.models.speech_encoder import SpeechEncoder
from src.models.transformer_fusion import TransformerFusionClassifier

cfg = load_config('config/default.yaml')
set_seed(cfg.seed)
paths = get_paths(cfg)
device = get_device()
ckpt = Path(paths['checkpoints'])

LABEL_NAMES = ['Angry', 'Happy', 'Sad', 'Neutral']
LABEL_EMOJI = ['😠', '😊', '😢', '😐']

print('Loading trained v4 models...\n')

# ── EEG Encoder ──
ecfg = cfg.model.eeg_encoder
eeg_enc = EEGEncoder(
    input_dim=ecfg.input_dim,
    hidden_dims=list(ecfg.hidden_dims),
    embedding_dim=ecfg.embedding_dim,
    dropout=ecfg.dropout,
).to(device)
eeg_source = 'random'
for name in ['v3/eeg_encoder_dann.pt', 'v3/eeg_encoder_contrastive.pt', 'eeg/eeg_encoder_final.pt']:
    if (ckpt / name).exists():
        try:
            eeg_enc.load_state_dict(torch.load(ckpt / name, map_location=device))
            eeg_source = name
            print(f'  ✅ EEG encoder  ← {name}')
            break
        except RuntimeError:
            continue
eeg_enc.eval()

# ── Speech Encoder ──
scfg = cfg.model.speech_encoder
speech_enc = SpeechEncoder(
    n_features=scfg.n_mfcc,
    embedding_dim=scfg.embedding_dim,
).to(device)
sp_source = 'random'
for name in ['v3/speech_encoder_dann.pt', 'v3/speech_encoder_contrastive.pt', 'speech/speech_encoder_final.pt']:
    if (ckpt / name).exists():
        try:
            speech_enc.load_state_dict(torch.load(ckpt / name, map_location=device))
            sp_source = name
            print(f'  ✅ Speech enc   ← {name}')
            break
        except RuntimeError:
            continue
speech_enc.eval()

# ── Transformer Fusion ──
v3 = getattr(cfg, 'v3', {})
tf = v3.get('transformer_fusion', {}) if isinstance(v3, dict) else getattr(v3, 'transformer_fusion', {})
_g = lambda d, k, default: d.get(k, default) if isinstance(d, dict) else getattr(d, k, default)

fusion = TransformerFusionClassifier(
    eeg_embed_dim=ecfg.embedding_dim,
    speech_embed_dim=scfg.embedding_dim,
    n_tokens=_g(tf, 'n_tokens', 8),
    d_model=_g(tf, 'd_model', 64),
    n_heads=_g(tf, 'n_heads', 4),
    n_layers=_g(tf, 'n_layers', 2),
    num_classes=cfg.model.num_classes,
    dropout=0.0,
    modality_dropout_prob=0.0,
).to(device)
fusion_source = 'random'
for name in ['v3/best_transformer_fusion.pt', 'v3/best_fusion_v3.pt']:
    if (ckpt / name).exists():
        try:
            sd = torch.load(ckpt / name, map_location=device)
            fusion.load_state_dict(sd.get('fusion', sd))
            fusion_source = name
            print(f'  ✅ Fusion       ← {name}')
            break
        except RuntimeError:
            continue
fusion.eval()

print(f'\n✅ All models loaded on {device}')

## 4. Load & Prepare Test Data

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  LOAD TEST DATA — DEAP (EEG) + IEMOCAP (Speech)              ║
# ║  Split with same seed as training → true held-out test set    ║
# ╚══════════════════════════════════════════════════════════════════╝
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from src.data.deap_loader import DEAPLoader
from src.data.iemocap_loader import IEMOCAPLoader

# Load raw features
deap = DEAPLoader(processed_dir=paths['deap_processed'])
eeg_feat, eeg_lbl, _ = deap.load_all(flatten=True)
iemocap = IEMOCAPLoader(processed_dir=paths['iemocap_processed'])
sp_feat, sp_lbl, _ = iemocap.load_all()

print(f'DEAP:    {len(eeg_feat)} samples, {eeg_feat.shape[1]}-dim features')
print(f'IEMOCAP: {len(sp_feat)} samples, {sp_feat.shape[1:]}-dim features')

# Same split as training (seed-locked)
_, eeg_Xv, _, eeg_yv = train_test_split(
    eeg_feat, eeg_lbl, test_size=0.2, stratify=eeg_lbl, random_state=cfg.seed)
_, sp_Xv, _, sp_yv = train_test_split(
    sp_feat, sp_lbl, test_size=0.2, stratify=sp_lbl, random_state=cfg.seed)

print(f'\nTest split: {len(eeg_Xv)} EEG, {len(sp_Xv)} speech')

# Encode
print('\nEncoding test data...')
with torch.no_grad():
    eeg_emb = []
    for i in range(0, len(eeg_Xv), 512):
        batch = torch.as_tensor(eeg_Xv[i:i+512], dtype=torch.float32).to(device)
        eeg_emb.append(eeg_enc(batch).cpu())
    eeg_emb = torch.cat(eeg_emb)

    sp_emb = []
    for i in range(0, len(sp_Xv), 512):
        batch = torch.as_tensor(sp_Xv[i:i+512], dtype=torch.float32).to(device)
        sp_emb.append(speech_enc(batch).cpu())
    sp_emb = torch.cat(sp_emb)

print(f'  EEG embeddings:    {eeg_emb.shape}')
print(f'  Speech embeddings: {sp_emb.shape}')

# Label-aligned pairing (same as evaluation)
eeg_yv_t = torch.as_tensor(eeg_yv, dtype=torch.long)
sp_yv_t = torch.as_tensor(sp_yv, dtype=torch.long)

eeg_list, sp_list, lbl_list = [], [], []
for c in range(4):
    eeg_c = eeg_emb[eeg_yv_t == c]
    sp_c = sp_emb[sp_yv_t == c]
    n = min(len(eeg_c), len(sp_c))
    if n == 0: continue
    perm_e = torch.randperm(len(eeg_c))[:n]
    perm_s = torch.randperm(len(sp_c))[:n]
    eeg_list.append(eeg_c[perm_e])
    sp_list.append(sp_c[perm_s])
    lbl_list.append(torch.full((n,), c, dtype=torch.long))

test_eeg = torch.cat(eeg_list)
test_sp = torch.cat(sp_list)
test_labels = torch.cat(lbl_list)

print(f'\n📊 Label-aligned test pairs: {len(test_labels)}')
for c in range(4):
    n = (test_labels == c).sum().item()
    print(f'   {LABEL_EMOJI[c]} {LABEL_NAMES[c]:8s}: {n}')
print('\n✅ Test data ready!')

## 5. 🎯 Live Predictions — Random Sample Demo
Picks N random test samples, shows true vs predicted with confidence bars.

**Re-run this cell to see different random samples each time!**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎯 LIVE PREDICTIONS — Random test samples with confidence   ║
# ║  Re-run this cell to get different random samples!            ║
# ╚══════════════════════════════════════════════════════════════════╝
import random
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

N_SAMPLES = 10  # change this to show more/fewer samples
COLORS = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

total = len(test_labels)
indices = random.sample(range(total), min(N_SAMPLES, total))

print(f"{'='*70}")
print(f"  🎯 LIVE PREDICTIONS — {len(indices)} Random Test Samples")
print(f"{'='*70}\n")

correct = 0
rows = (len(indices) + 4) // 5
fig, axes = plt.subplots(rows, 5, figsize=(20, 4 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

# Hide unused axes
for ax in axes[len(indices):]:
    ax.set_visible(False)

for idx_i, sample_idx in enumerate(indices):
    eeg_sample = test_eeg[sample_idx:sample_idx+1].to(device)
    sp_sample = test_sp[sample_idx:sample_idx+1].to(device)
    true_label = test_labels[sample_idx].item()

    with torch.no_grad():
        logits = fusion(eeg_sample, sp_sample)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = int(logits.argmax(1).item())

    is_correct = pred == true_label
    correct += int(is_correct)
    status = '✅' if is_correct else '❌'

    print(f"  Sample {idx_i+1:2d} │ True: {LABEL_EMOJI[true_label]} {LABEL_NAMES[true_label]:8s} │ "
          f"Pred: {LABEL_EMOJI[pred]} {LABEL_NAMES[pred]:8s} │ "
          f"Conf: {probs[pred]*100:.1f}% │ {status}")

    # Bar chart
    ax = axes[idx_i]
    bars = ax.barh(LABEL_NAMES, probs * 100, color=COLORS, edgecolor='white', height=0.6)
    ax.set_xlim(0, 110)
    ax.set_title(f"#{idx_i+1}: True={LABEL_NAMES[true_label]}", fontsize=10,
                 fontweight='bold', color='green' if is_correct else 'red')
    bars[pred].set_edgecolor('black')
    bars[pred].set_linewidth(2)
    for i, v in enumerate(probs * 100):
        ax.text(v + 1.5, i, f"{v:.1f}%", va='center', fontsize=8)
    ax.tick_params(axis='y', labelsize=9)
    ax.tick_params(axis='x', labelsize=8)

demo_acc = correct / len(indices) * 100
print(f"\n  ── Demo accuracy: {correct}/{len(indices)} = {demo_acc:.0f}% ──")

plt.suptitle(f"AMERS v4 — Live Predictions ({correct}/{len(indices)} = {demo_acc:.0f}%)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AMERS/outputs/v4_demo_predictions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('\n📁 Saved to Drive: outputs/v4_demo_predictions.png')

## 6. 📈 Full Test Set Evaluation

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📈 FULL TEST SET — Accuracy, per-class, confidence, CM      ║
# ╚══════════════════════════════════════════════════════════════════╝
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, cohen_kappa_score

total = len(test_labels)

with torch.no_grad():
    all_preds, all_probs = [], []
    for i in range(0, total, 512):
        end = min(i + 512, total)
        logits = fusion(test_eeg[i:end].to(device), test_sp[i:end].to(device))
        all_preds.append(logits.argmax(1).cpu())
        all_probs.append(torch.softmax(logits, dim=1).cpu())
    preds = torch.cat(all_preds).numpy()
    probs = torch.cat(all_probs).numpy()

labels = test_labels.numpy()
acc = (preds == labels).mean()
f1_mac = f1_score(labels, preds, average='macro', zero_division=0)
f1_wt = f1_score(labels, preds, average='weighted', zero_division=0)
kappa = cohen_kappa_score(labels, preds)

print(f"{'='*70}")
print(f"  📈 AMERS v4 — FULL TEST SET RESULTS ({total} samples)")
print(f"{'='*70}")
print(f"\n  Overall Accuracy:  {acc*100:.2f}%")
print(f"  Macro F1:          {f1_mac:.4f}")
print(f"  Weighted F1:       {f1_wt:.4f}")
print(f"  Cohen's Kappa:     {kappa:.4f}")

# Per-class
print(f"\n  {'─'*60}")
print(f"  Per-class breakdown:")
print(f"  {'─'*60}")
for c in range(4):
    mask = labels == c
    if mask.sum() == 0: continue
    cls_acc = (preds[mask] == c).mean()
    avg_conf = probs[mask, c].mean()
    print(f"    {LABEL_EMOJI[c]} {LABEL_NAMES[c]:8s} │ "
          f"Acc: {cls_acc*100:5.1f}% │ "
          f"Avg conf: {avg_conf*100:5.1f}% │ "
          f"n={mask.sum()}")

# Confidence analysis
pred_confs = probs[np.arange(len(preds)), preds]
correct_mask = preds == labels
print(f"\n  {'─'*60}")
print(f"  Confidence analysis:")
print(f"  {'─'*60}")
print(f"    When CORRECT:  avg conf = {pred_confs[correct_mask].mean()*100:.1f}%")
if (~correct_mask).sum() > 0:
    print(f"    When WRONG:    avg conf = {pred_confs[~correct_mask].mean()*100:.1f}%")
print(f"    Overall avg:   avg conf = {pred_confs.mean()*100:.1f}%")

# Classification report
print(f"\n  {'─'*60}")
print(f"  Classification Report:")
print(f"  {'─'*60}")
print(classification_report(labels, preds, target_names=LABEL_NAMES, digits=4))

# Confusion matrix
cm = confusion_matrix(labels, preds)
print(f"  Confusion Matrix (rows=true, cols=predicted):")
print(f"           {'  '.join(f'{n:>7s}' for n in LABEL_NAMES)}")
for i in range(4):
    row = '  '.join(f'{cm[i,j]:7d}' for j in range(4))
    print(f"    {LABEL_NAMES[i]:8s} {row}")

print(f"\n{'='*70}")
print(f"  🏆 Model is working! Accuracy: {acc*100:.2f}%, Macro F1: {f1_mac:.4f}")
print(f"{'='*70}")

## 7. 📊 Visual Confusion Matrix & Confidence Distribution

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📊 VISUAL PLOTS — Confusion matrix heatmap + confidence     ║
# ╚══════════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── 1. Confusion Matrix Heatmap ──
ax = axes[0]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(LABEL_NAMES, fontsize=10)
ax.set_yticks(range(4)); ax.set_yticklabels(LABEL_NAMES, fontsize=10)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title('Normalized Confusion Matrix', fontsize=12, fontweight='bold')
for i in range(4):
    for j in range(4):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                ha='center', va='center', fontsize=9, color=color)
plt.colorbar(im, ax=ax, fraction=0.046)

# ── 2. Confidence Distribution ──
ax = axes[1]
correct_confs = pred_confs[correct_mask]
wrong_confs = pred_confs[~correct_mask]
ax.hist(correct_confs * 100, bins=30, alpha=0.7, color='#2ecc71', label=f'Correct (n={len(correct_confs)})', edgecolor='white')
if len(wrong_confs) > 0:
    ax.hist(wrong_confs * 100, bins=30, alpha=0.7, color='#e74c3c', label=f'Wrong (n={len(wrong_confs)})', edgecolor='white')
ax.set_xlabel('Confidence (%)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Confidence Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)

# ── 3. Per-Class Accuracy Bar Chart ──
ax = axes[2]
cls_accs = []
for c in range(4):
    mask = labels == c
    cls_accs.append((preds[mask] == c).mean() * 100 if mask.sum() > 0 else 0)
bars = ax.bar(LABEL_NAMES, cls_accs, color=COLORS, edgecolor='white', width=0.6)
ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Per-Class Accuracy', fontsize=12, fontweight='bold')
for bar, v in zip(bars, cls_accs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1.5, f'{v:.1f}%',
            ha='center', fontsize=10, fontweight='bold')
ax.axhline(y=acc*100, color='black', linestyle='--', alpha=0.4, label=f'Overall: {acc*100:.1f}%')
ax.legend(fontsize=10)

plt.suptitle(f'AMERS v4 — Test Set Analysis (Acc={acc*100:.1f}%, F1={f1_mac:.3f})',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AMERS/outputs/v4_test_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved to Drive: outputs/v4_test_analysis.png')

## 8. 🔬 Single Sample Deep Dive
Change `SAMPLE_INDEX` below to inspect any specific test sample in detail.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔬 SINGLE SAMPLE DEEP DIVE — inspect one prediction         ║
# ║  Change SAMPLE_INDEX to look at different samples             ║
# ╚══════════════════════════════════════════════════════════════════╝
import random

SAMPLE_INDEX = random.randint(0, len(test_labels) - 1)  # or set manually: e.g., 42

eeg_sample = test_eeg[SAMPLE_INDEX:SAMPLE_INDEX+1].to(device)
sp_sample = test_sp[SAMPLE_INDEX:SAMPLE_INDEX+1].to(device)
true_label = test_labels[SAMPLE_INDEX].item()

with torch.no_grad():
    logits = fusion(eeg_sample, sp_sample)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(logits.argmax(1).item())

is_correct = pred == true_label

print(f"{'='*50}")
print(f"  🔬 Single Sample Deep Dive — #{SAMPLE_INDEX}")
print(f"{'='*50}")
print(f"\n  True emotion:      {LABEL_EMOJI[true_label]} {LABEL_NAMES[true_label]}")
print(f"  Predicted emotion: {LABEL_EMOJI[pred]} {LABEL_NAMES[pred]}")
print(f"  Result:            {'✅ CORRECT' if is_correct else '❌ WRONG'}")
print(f"\n  Confidence scores:")
for c in range(4):
    bar = '█' * int(probs[c] * 40)
    marker = ' ◄── PRED' if c == pred else (' ◄── TRUE' if c == true_label and not is_correct else '')
    print(f"    {LABEL_EMOJI[c]} {LABEL_NAMES[c]:8s} {probs[c]*100:5.1f}% {bar}{marker}")

# Logits (raw)
logits_np = logits.cpu().numpy()[0]
print(f"\n  Raw logits: {', '.join(f'{v:.3f}' for v in logits_np)}")

# EEG embedding stats
eeg_np = eeg_sample.cpu().numpy()[0]
sp_np = sp_sample.cpu().numpy()[0]
print(f"\n  EEG embedding:    mean={eeg_np.mean():.4f}, std={eeg_np.std():.4f}, "
      f"min={eeg_np.min():.4f}, max={eeg_np.max():.4f}")
print(f"  Speech embedding: mean={sp_np.mean():.4f}, std={sp_np.std():.4f}, "
      f"min={sp_np.min():.4f}, max={sp_np.max():.4f}")
print(f"{'='*50}")